In [67]:



names = [
    "Alba",
    "Ari",
    "Paolo",
    "Dani",
    "Grazia",
    "Moreno",
    "Edo",
    "Flavio",
    "Sonia",
    "Pippo",
    "Marci",

    "Andrea",
    "Silvia",
    "Giada",
    "Johnny",

    "Nonna Ilda",
    "Nonna Elena"
]




codes_unshuffled = [
    "PK94", "GT33", "MN23", "PL87", "RC21", "VR46",
    "PC45", "RM28", "FE12", "CT36", "FI09",
    "TN57", "BS78", "NA34", "TO67",
    "MD85", "ME94"
]


import random

# shuffle the codes
codes_shuffled = codes_unshuffled[:]
random.shuffle(codes_shuffled)



In [68]:
# create a dataframe from the dictionary
import pandas as pd
df = pd.DataFrame(names, columns=["name"])
df["code"] = codes_shuffled

In [69]:

names_last_year_str = "Sonia>Nonna Ilda>Silvia>Pippo>Paolo>Edo>Flavio>Moreno>Nonna Elena>Grazia>Ari>Johnny>Andrea>Dani>Marci>Giada>Alba "
names_last_year = [name.strip() for name in names_last_year_str.split(">")]
print(names_last_year)
print(len(names_last_year))

['Sonia', 'Nonna Ilda', 'Silvia', 'Pippo', 'Paolo', 'Edo', 'Flavio', 'Moreno', 'Nonna Elena', 'Grazia', 'Ari', 'Johnny', 'Andrea', 'Dani', 'Marci', 'Giada', 'Alba']
17


In [70]:
def create_cycle(names, last_year):
    n = len(names)
    index_map = {name: i for i, name in enumerate(names)}
    last_year_indices = [index_map[name] for name in last_year]

    cycle = [0] * n
    for i in range(n):
        current_index = last_year_indices[i]
        next_index = last_year_indices[(i + 1) % n]
        cycle[current_index] = next_index

    return [names[i] for i in cycle]


last_year = create_cycle(names, names_last_year)
df["last_year"] = last_year

In [71]:
import random

def single_cycle_list(lst):
    lst = lst[:]        # copy
    random.shuffle(lst) # shuffle to avoid trivial cycles
    return lst          # this list *is* the cycle order
                        # lst[i] → lst[i+1], last → first


this_year= single_cycle_list(names)
df["this_year"] = this_year
print(this_year)

['Silvia', 'Pippo', 'Moreno', 'Nonna Ilda', 'Sonia', 'Flavio', 'Nonna Elena', 'Edo', 'Dani', 'Ari', 'Grazia', 'Alba', 'Paolo', 'Johnny', 'Andrea', 'Giada', 'Marci']


In [72]:


while True:
    if all(df["this_year"][i] != df["last_year"][i] for i in range(len(names))):
        break
    this_year= single_cycle_list(names)
    df["this_year"] = this_year
    print("Regenerating...")

Regenerating...


In [73]:
df

,name,code,last_year,this_year
0,Alba,BS78,Sonia,Dani
1,Ari,FE12,Johnny,Silvia
2,Paolo,TO67,Edo,Moreno
3,Dani,TN57,Marci,Edo
4,Grazia,MN23,Ari,Grazia
5,Moreno,GT33,Nonna Elena,Nonna Ilda
6,Edo,MD85,Flavio,Pippo
7,Flavio,RC21,Moreno,Johnny
8,Sonia,RM28,Nonna Ilda,Paolo
9,Pippo,VR46,Paolo,Nonna Elena


In [74]:
# read frpom the code using input
input_code = input("Enter your code: ").strip()



In [75]:
matched_names = df[df["code"] == input_code]["name"].tolist()
if matched_names:
    print(f"Ciao: {', '.join(matched_names)}, devi fare un regalo a {df[df['name'] == matched_names[0]]['this_year'].values[0]}!")


Ciao: Moreno, devi fare un regalo a Nonna Ilda!


In [76]:
import json

# Build assignment dictionary
assignment_dict = {}

for i, row in df.iterrows():
    code = row["code"]
    name = row["name"]
    giftee = row["this_year"]
    assignment_dict[code] = {
        "name": name,
        "giftee": giftee
    }

# (Optional) Encode giftees to Base64 to hide them
import base64
encoded_dict = {
    code: {
        "name": data["name"],
        "giftee": base64.b64encode(data["giftee"].encode()).decode()
    }
    for code, data in assignment_dict.items()
}

with open("assignments.json", "w") as f:
    json.dump(encoded_dict, f)


<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<title>Secret Santa 2025</title>
<style>
body { font-family: Arial; max-width: 500px; margin: 40px auto; }
input,button { font-size: 18px; padding: 6px; }
#result { margin-top: 20px; font-size: 20px; font-weight: bold; }
</style>
</head>
<body>

<h2>Secret Santa — Inserisci il tuo codice</h2>

<input id="codeInput" type="text" placeholder="Es. AB10">
<button onclick="checkCode()">Mostra destinatario</button>

<div id="result"></div>

<script>
// Load JSON once at page load
let assignments = {};

fetch("assignments.json")
    .then(response => response.json())
    .then(data => assignments = data);

function decodeBase64(str) {
    return atob(str);
}

function checkCode() {
    const code = document.getElementById("codeInput").value.trim();
    const result = document.getElementById("result");

    if (!assignments[code]) {
        result.innerHTML = "<span style='color:red'>Codice non valido 😢</span>";
        return;
    }

    const person = assignments[code]["name"];
    const giftee = decodeBase64(assignments[code]["giftee"]);

    result.innerHTML = `Ciao <b>${person}</b>! 🎁 Devi fare un regalo a <b style="color:green">${giftee}</b>!`;
}
</script>

</body>
</html>
